# Factor Investing & Systematic Portfolio Construction

This notebook analyses the results of a systematic factor-based equity portfolio built on the S&P 500 universe.

The portfolio is constructed using the Fama-French four-factor model (MKT, SMB, HML, Momentum).
Factor loadings are estimated via rolling OLS regression. Expected return signals are derived from
rolling factor premia. Portfolio weights are optimised monthly using cvxpy under realistic constraints:
maximum weight per asset (5%), tracking error limit (4% annualised), and turnover limit (20% per month).

The analysis covers:
1. Performance attribution via factor regression
2. Cumulative returns vs S&P 500 benchmark
3. Drawdown analysis
4. Rolling Sharpe ratio
5. Factor exposure summary

In [1]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
sys.path.append("..")
import pandas as pd
from statsmodels.regression.linear_model import OLS
from src.plot import cumulative_returns, drawdown_plot, rolling_sharpe_plot, factor_loadings_plot
from config import rf

## Load results

We load the pre-computed outputs from main.py:
portfolio returns (net of transaction costs), weights, turnover, rolling Sharpe ratio,
drawdown, and the factor series used for attribution.

The S&P 500 return series serves as the benchmark throughout the analysis.

In [2]:
BASE_DIR = Path().resolve().parent
portfolio_path = BASE_DIR / "results" / "portfolio"
metrics_path = BASE_DIR / "results" / "metrics"
data_path = BASE_DIR / "results" / "data"
prices_path = BASE_DIR / "results" / "prices"

portfolio_returns = pd.read_parquet(portfolio_path / "portfolio_returns.parquet").squeeze()
sp500 = pd.read_parquet(prices_path / "sp500_returns.parquet").squeeze()
portfolio_weights = pd.read_parquet(portfolio_path / "portfolio_weights.parquet").squeeze()
turnover = pd.read_parquet(metrics_path / "turnover.parquet").squeeze()
sharpe_ratio = pd.read_parquet(metrics_path / "sharpe_ratio.parquet").squeeze()
drawdown = pd.read_parquet(metrics_path / "drawdown.parquet").squeeze()
information_raio = pd.read_parquet(metrics_path / "information_ratio.parquet").squeeze()
x = pd.read_parquet(data_path / "factors.parquet")
x = x.loc[portfolio_returns.index]
sp500 = sp500.loc[portfolio_returns.index]

## Performance attribution — OLS factor regression

We regress the portfolio excess returns (portfolio return minus daily risk-free rate)
on the four Fama-French factors to decompose performance into systematic and idiosyncratic components.

The regression follows the standard factor model:

    r_p - r_f = alpha + beta_MKT * MKT + beta_SMB * SMB + beta_HML * HML + beta_MOM * MOM + epsilon

A statistically significant positive alpha would indicate excess return beyond factor compensation.
An alpha close to zero indicates that performance is fully explained by systematic factor exposures.

In [3]:
y = portfolio_returns - rf
x["alpha"] = 1
alpha = OLS(y, x).fit()
print(alpha.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.948
Model:                            OLS   Adj. R-squared:                  0.947
Method:                 Least Squares   F-statistic:                     8930.
Date:                Mon, 11 May 2026   Prob (F-statistic):               0.00
Time:                        03:12:23   Log-Likelihood:                 8510.1
No. Observations:                1984   AIC:                        -1.701e+04
Df Residuals:                    1979   BIC:                        -1.698e+04
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Mkt-RF         1.1159      0.006    180.379      0.0

### Interpretation

The regression explains 94.8% of portfolio return variance (R² = 0.948), confirming that the
portfolio is predominantly driven by systematic factor exposures.

Key findings:

- **Market beta = 1.116**: the portfolio carries slightly more market risk than the benchmark,
  consistent with an active tilt away from the equal-weight benchmark.

- **SMB = -0.053**: mild large-cap bias, expected given the S&P 500 universe which excludes small caps.

- **HML = -0.110**: negative value loading, indicating a tilt toward growth stocks. This is consistent
  with the momentum signal dominating the optimisation in recent years, where growth outperformed value.

- **Momentum = 0.123**: significant positive momentum exposure, statistically significant at all
  conventional levels. The portfolio systematically overweights recent winners.

- **Alpha = -6.8e-05, p-value = 0.361**: not statistically significant. The portfolio does not
  generate returns beyond what is explained by factor exposures. This is consistent with the
  semi-strong form efficiency of large-cap US equity markets and the use of publicly available data.

The absence of alpha is an honest and expected result — not a failure of the strategy.
It confirms that the portfolio's returns are a fair compensation for systematic risk exposures,
not luck or data mining.

## Visual analysis

We produce four charts to summarise portfolio behaviour:

1. **Cumulative returns**: factor portfolio vs S&P 500 — overall performance over the sample period
2. **Drawdown**: peak-to-trough losses — behaviour during market stress (2020 COVID, 2022 rate shock)
3. **Rolling Sharpe ratio**: risk-adjusted performance stability over time
4. **Factor loadings**: average portfolio exposure to each systematic factor

In [4]:
portfolio_loadings = pd.Series({
    "Mkt-RF": 1.1159,
    "SMB": -0.0526,
    "HML": -0.1099,
    "Mom": 0.1226
})
cumulative_returns(portfolio_returns, sp500)
drawdown_plot(drawdown, sp500)
rolling_sharpe_plot(sharpe_ratio)
factor_loadings_plot(portfolio_loadings)

In [5]:
print(f"Annualised Sharpe ratio (mean): {sharpe_ratio.mean():.3f}")
print(f"Maximum drawdown: {drawdown.min():.3%}")
print(f"Average Information Ratio: {information_raio.mean():.3f}")
print(f"Average monthly turnover: {turnover.mean():.3%}")
print(f"Net annualised return: {portfolio_returns.mean() * 252:.3%}")
print(f"Net annualised volatility: {portfolio_returns.std() * (252**0.5):.3%}")

Annualised Sharpe ratio (mean): 0.670
Maximum drawdown: -31.379%
Average Information Ratio: 0.731
Average monthly turnover: 0.878%
Net annualised return: 15.888%
Net annualised volatility: 22.997%


## Summary statistics

Key performance metrics over the full out-of-sample period.

## Summary and conclusions

The factor portfolio is a systematic, rules-based strategy built on well-documented risk premia.
Over the 2018–2026 sample period, the main findings are:

- The portfolio tracks the S&P 500 closely, consistent with the 4% tracking error constraint.
- Performance is almost entirely explained by four systematic factors (R² = 0.948).
- No statistically significant alpha is generated — consistent with large-cap market efficiency.
- The dominant exposures are market risk (beta > 1) and momentum (positive loading).
- The negative HML loading reflects a growth tilt, which has been rewarded in the sample period.

**Key limitation**: the strategy uses the current S&P 500 composition, introducing survivorship bias.
Companies that were delisted or went bankrupt during the sample period are not included,
which likely overstates backtest performance.

**On tracking error**: the 4% annualised tracking error constraint proved tight for this universe.
The optimiser operates near the constraint boundary in most periods, limiting the portfolio's
ability to deviate from the market-cap benchmark. A less constrained version would likely
show stronger factor tilts and potentially higher active return.

For further details on methodology, see docs/technical_report.pdf.